# Phase 1: Machine Learning Model Benchmark
### ML-Assisted IMU Drift Reduction for GNSS-Denied UAV Navigation

This notebook demonstrates:
1. Loading the telemetry dataset and generating 142 leakage-safe IMU features
2. Applying whole-flight log splitting (`log_51`+`log_53` train, `log_52` val, `log_54` test)
3. Evaluating Ridge Regression, Random Forest, and Gradient Boosting
4. Analyzing velocity residual predictions across North, East, and Down axes
5. Inspecting Random Forest feature importances

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in python path
sys.path.insert(0, os.path.abspath('..'))

from models.utils import load_dataset, log_based_split, compute_metrics, TARGET_COLUMNS, TARGET_LABELS
from feature_engineering import engineer_features, get_feature_columns

## 1. Load Dataset & Feature Engineering Pipeline

In [ ]:
df_raw = load_dataset('../imu_drift_feature_dataset.csv')
print(f"Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# Run leakage-safe temporal feature engineering
df_featured = engineer_features(df_raw, verbose=True)
feature_cols = get_feature_columns(df_featured)
print(f"Generated {len(feature_cols)} input features")

## 2. Log-Based Partitioning

In [ ]:
df_train, df_val, df_test = log_based_split(df_featured)

X_test = df_test[feature_cols].values
y_test = df_test[TARGET_COLUMNS].values
t_test = df_test['time_s'].values
print(f"Test set (log_54): {len(df_test):,} samples")

## 3. Load Trained Model Checkpoints & Evaluate

In [ ]:
import pickle

models = {}
scalers = {}
for name, m_path, s_path in [
    ('Ridge', '../saved_models/ridge_model.pkl', '../saved_models/ridge_scaler.pkl'),
    ('Random Forest', '../saved_models/rf_model.pkl', '../saved_models/rf_scaler.pkl'),
    ('Gradient Boosting', '../saved_models/gbm_model.pkl', '../saved_models/gbm_scaler.pkl')
]:
    with open(m_path, 'rb') as f:
        models[name] = pickle.load(f)
    with open(s_path, 'rb') as f:
        scalers[name] = pickle.load(f)

results = []
predictions = {}
for name, model in models.items():
    X_s = scalers[name].transform(X_test)
    y_pred = model.predict(X_s)
    predictions[name] = y_pred
    m = compute_metrics(y_test, y_pred)
    m['Model'] = name
    results.append(m)

df_results = pd.DataFrame(results).sort_values('rmse_mean')
display(df_results[['Model', 'rmse_mean', 'mae_mean', 'r2_mean', 'rmse_N', 'rmse_E', 'rmse_D']])

## 4. Visual Comparison: Prediction vs Ground Truth Time Series

In [ ]:
n_pts = 1000
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
colors = {'Ridge': 'crimson', 'Random Forest': 'forestgreen', 'Gradient Boosting': 'darkorange'}

for i, (ax, label) in enumerate(zip(axes, TARGET_LABELS)):
    ax.plot(t_test[:n_pts], y_test[:n_pts, i], label='True Residual', color='black', lw=1.2)
    for name, y_pred in predictions.items():
        ax.plot(t_test[:n_pts], y_pred[:n_pts, i], label=name, color=colors[name], lw=0.9, ls='--')
    ax.set_ylabel(f'{label} (m/s)', fontsize=11)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(loc='upper right', ncol=4)

axes[-1].set_xlabel('Flight Time (s)', fontsize=11)
plt.suptitle('Multi-Model Velocity Drift Residual Prediction (log_54)', fontsize=13, y=0.99)
plt.tight_layout()
plt.show()

## 5. Feature Importance Analysis (Random Forest)

In [ ]:
rf_fi = pd.read_csv('../results/rf_feature_importances.csv')

plt.figure(figsize=(10, 6))
sns.barplot(data=rf_fi.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Most Important IMU Features for Drift Prediction (RF Gini Importance)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()